In [1]:
import datetime
import datetime as dt

import numpy as np
from matplotlib import pyplot as plt
from netCDF4 import Dataset
import pandas as pd
from scipy.optimize import minimize
import xarray as xr

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset

In [2]:
from dask.distributed import Client, LocalCluster
import dask

cluster = LocalCluster(
    n_workers=14,              
    threads_per_worker=2,
    memory_limit='2GB', # per worker
)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 14
Total threads: 28,Total memory: 26.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44709,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38793,Total threads: 2
Dashboard: http://127.0.0.1:38543/status,Memory: 1.86 GiB
Nanny: tcp://127.0.0.1:34881,


2025-11-03 11:05:10,858 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='deep.earthsystemdatalab.net', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/conda/deepesdl54/4162d0b2-1760697589-145-irrigation-eu/lib/python3.13/site-packages/tornado/websocket.py", line 938, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/conda/deepesdl54/4162d0b2-1760697589-145-irrigation-eu/lib/python3.13/site-packages/tornado/web.py", line 3301, in wrapper
    return method(self, *args, **kwargs)
  File "/home/conda/deepesdl54/4162d0b2-1760697589-145-irrigation-eu/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: T

In [3]:
import os

S3_USER_STORAGE_KEY = os.environ["S3_USER_STORAGE_KEY"]
S3_USER_STORAGE_SECRET = os.environ["S3_USER_STORAGE_SECRET"]
S3_USER_STORAGE_BUCKET = os.environ["S3_USER_STORAGE_BUCKET"]

irr_store = new_data_store("s3",
                       root=S3_USER_STORAGE_BUCKET,
                       storage_options=dict(anon=False,
                                            key=S3_USER_STORAGE_KEY,
                                            secret=S3_USER_STORAGE_SECRET))

In [4]:
irr_store.list_data_ids()

['calibrated.zarr',
 'calibrated_0.zarr',
 'calibrated_1.zarr',
 'calibrated_10.zarr',
 'calibrated_11.zarr',
 'calibrated_12.zarr',
 'calibrated_13.zarr',
 'calibrated_14.zarr',
 'calibrated_15.zarr',
 'calibrated_16.zarr',
 'calibrated_17.zarr',
 'calibrated_18.zarr',
 'calibrated_19.zarr',
 'calibrated_2.zarr',
 'calibrated_20.zarr',
 'calibrated_3.zarr',
 'calibrated_4.zarr',
 'calibrated_5.zarr',
 'calibrated_6.zarr',
 'calibrated_7.zarr',
 'calibrated_8.zarr',
 'calibrated_9.zarr',
 'deleteme.zarr',
 'era5.zarr',
 'era5v2.zarr',
 'era5v3_timeopt.zarr',
 'irrigation_input.zarr',
 'irrigation_input_small_chunks.zarr',
 'iwu_estimates.zarr',
 'iwu_estimates_temporal.zarr',
 'soil_moisture_filled.zarr',
 'soil_moisture_zappend.zarr']

In [5]:
ds = irr_store.open_data("irrigation_input_small_chunks.zarr")
ds

<xarray.Dataset> Size: 1TB
Dimensions:      (time: 3561, lat: 4144, lon: 6832)
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Data variables:
    SWI          (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>
    pev          (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>
    tp           (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>

In [6]:
calibration = irr_store.open_data("calibrated.zarr")
calibration

<xarray.Dataset> Size: 906MB
Dimensions:      (lat: 4144, lon: 6832, params: 4)
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * params       (params) <U2 32B 'a' 'b' 'z' 'RF'
    spatial_ref  int64 8B ...
Data variables:
    calibration  (lat, lon, params) float64 906MB dask.array<chunksize=(50, 50, 4), meta=np.ndarray>

In [7]:
def ts_smet4irr(sm, et, a, b, z, RF, thr=None):
    p_sim = z * (sm[1:] - sm[:-1]) \
          + ((a * sm[1:]**b + a * sm[:-1]**b) / 2.) \
          + ((RF * sm[1:] * et[1:] + RF * sm[:-1] * et[:-1]) / 2.)

    p_sim[abs(np.diff(sm)) <= 0.001] = 0.0
    p_sim[p_sim < 1.0] = 0.0
    return np.clip(p_sim, 0, thr)


psim = xr.apply_ufunc(
    ts_smet4irr,
    ds["SWI"],
    ds["pev"],
    calibration["calibration"].sel(params="a"),
    calibration["calibration"].sel(params="b"),
    calibration["calibration"].sel(params="z"),
    calibration["calibration"].sel(params="RF"),
    input_core_dims=[["time"], ["time"], [], [], [], []],
    output_core_dims=[["time2"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
    dask_gufunc_kwargs={"output_sizes": {"time2": ds.sizes["time"] - 1}}
)


In [8]:
psim

<xarray.DataArray (lat: 4144, lon: 6832, time2: 3560)> Size: 806GB
dask.array<transpose, shape=(4144, 6832, 3560), dtype=float64, chunksize=(50, 50, 3560), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0
Dimensions without coordinates: time2

In [9]:
psim2 = psim.rename({"time2": "time"})
time_coord = ds["time"].values[:-1] # TODO: remove the last element
psim2 = psim2.assign_coords(time=time_coord)
psim2 = psim2.transpose("time", "lat", "lon")
psim2

<xarray.DataArray (time: 3560, lat: 4144, lon: 6832)> Size: 806GB
dask.array<transpose, shape=(3560, 4144, 6832), dtype=float64, chunksize=(3560, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-29
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0

In [10]:
# First aggregate both psim2 and ds[tp] weekly sum()

# because psim2 is 1 timestamp less than ds['tp'] and the last value is removed during the calcuation of psim2, we need to remove 
# the last timestep here as well so they have the same shape.
tp_except_last_timestamp = ds['tp'].isel(time=slice(0, -1)) 
tp_except_last_timestamp

<xarray.DataArray 'tp' (time: 3560, lat: 4144, lon: 6832)> Size: 403GB
dask.array<getitem, shape=(3560, 4144, 6832), dtype=float32, chunksize=(3560, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-29
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Attributes: (12/33)
    GRIB_NV:                                  0
    GRIB_Nx:                                  611
    GRIB_Ny:                                  373
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            fc
    ...                                       ...
    GRIB_units:                               mm
    GRIB_uvRelativeToGrid:                    0
    grid_mapping:                             spatial_ref
    long_name:                                Total precipitation (millimeters)
    standard_name:                            unknown
    units:                                    mm

In [11]:
def resample_sum(dataarray: xr.DataArray, step: int) -> xr.DataArray:
    data = dataarray.data
    steps = dataarray.time.size // step
    data = data[:steps*step, :, :]
    data = data.reshape(steps, step, dataarray.sizes["lat"], dataarray.sizes["lon"])
    data_sum = data.sum(axis=1)
    return xr.DataArray(
        data=data_sum,
        dims=("time", "lat", "lon"),
        coords=dict(
            time=dataarray.time[::step][:steps],
            lat=dataarray.lat,
            lon=dataarray.lon,
            spatial_ref=dataarray.spatial_ref,
        ),
        attrs=dataarray.attrs,
    )

In [12]:
tp_weekly = resample_sum(tp_except_last_timestamp, step=7)
tp_weekly

<xarray.DataArray 'sum-aggregate-259ce54931cdbfc08e821be7bbf8baf0' (time: 508,
                                                                    lat: 4144,
                                                                    lon: 6832)> Size: 58GB
dask.array<sum-aggregate, shape=(508, 4144, 6832), dtype=float32, chunksize=(508, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 4kB 2016-01-01 2016-01-08 ... 2025-09-19
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Attributes: (12/33)
    GRIB_NV:                                  0
    GRIB_Nx:                                  611
    GRIB_Ny:                                  373
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            fc
    ...                                       ...
    GRIB_units:                               mm
    GRIB_uvRelativeToGrid:                    0
    grid_mapping:                             spatial_ref
    long_name:                                Total precipitation (millimeters)
    standard_name:                            unknown
    units:                                    mm

In [13]:
psim2_weekly = resample_sum(psim2, step=7)
psim2_weekly

<xarray.DataArray 'sum-aggregate-4db81ab8a59003890d74098e3140d305' (time: 508,
                                                                    lat: 4144,
                                                                    lon: 6832)> Size: 115GB
dask.array<sum-aggregate, shape=(508, 4144, 6832), dtype=float64, chunksize=(508, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 4kB 2016-01-01 2016-01-08 ... 2025-09-19
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0

In [14]:
IRR = psim2_weekly - tp_weekly

IRR_clipped = IRR.clip(0, 1000)

IRR_weekly = IRR_clipped.where(IRR_clipped/tp_weekly >= 0.2, 0)

IRR_weekly

<xarray.DataArray (time: 508, lat: 4144, lon: 6832)> Size: 115GB
dask.array<where, shape=(508, 4144, 6832), dtype=float64, chunksize=(508, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 4kB 2016-01-01 2016-01-08 ... 2025-09-19
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0

In [15]:
IRR_biweekly = resample_sum(IRR_weekly, step=2)
IRR_biweekly

<xarray.DataArray 'sum-aggregate-3f748a9ea79e1fde613e36383e5863e6' (time: 254,
                                                                    lat: 4144,
                                                                    lon: 6832)> Size: 58GB
dask.array<sum-aggregate, shape=(254, 4144, 6832), dtype=float64, chunksize=(254, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 2kB 2016-01-01 2016-01-15 ... 2025-09-12
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0

In [16]:
%%time
irr_store.write_data(IRR_biweekly.to_dataset(name="iwu_est"), "iwu_estimates_temporal.zarr", replace=True)

/home/conda/deepesdl54/4162d0b2-1760697589-145-irrigation-eu/lib/python3.13/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 9.76 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


CPU times: user 6min 1s, sys: 46.4 s, total: 6min 48s
Wall time: 26min 26s


'iwu_estimates_temporal.zarr'

In [21]:
IRR_biweekly_temporal = irr_store.open_data("iwu_estimates_temporal.zarr")
IRR_biweekly_spatial = chunk_dataset(IRR_biweekly_temporal,  {"time":1, "lat": 2072, "lon": 1708}, format_name="zarr")
IRR_biweekly_spatial

<xarray.Dataset> Size: 58GB
Dimensions:      (time: 254, lat: 4144, lon: 6832)
Coordinates:
  * time         (time) datetime64[ns] 2kB 2016-01-01 2016-01-15 ... 2025-09-12
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Data variables:
    iwu_est      (time, lat, lon) float64 58GB dask.array<chunksize=(1, 2072, 1708), meta=np.ndarray>

In [22]:
%%time
irr_store.write_data(IRR_biweekly_spatial, "iwu_estimates_spatial.zarr", replace=True)

2025-11-03 11:37:50,690 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 1.12 GiB -- Worker memory limit: 1.86 GiB
2025-11-03 11:37:52,227 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 1.20 GiB -- Worker memory limit: 1.86 GiB


CPU times: user 1min 33s, sys: 11.9 s, total: 1min 45s
Wall time: 8min 23s


'iwu_estimates_spatial.zarr'

In [23]:
client.shutdown()